# 01 — How isodepth is built, and what it is on each cell

**Claim under test:** the mouse/human spatial coordinate `isodepth_consensus` is anchored in real
tissue geometry, and we know exactly which cells carry a *fit* vs a *smoothed* value.

Two layers, and conflating them caused a real error in this project:

| layer | what it is | file |
|---|---|---|
| per-sample GASTON fit | radius/isodepth fit **per section** | `crossspecies_radial_gp/transfer_all/radius_*.npz` |
| consensus | seeded from a **subset** of those, then graph-smoothed to unseeded cells | `results/isodepth/*_fibdec_isodepth.h5ad` |

`uninduced` **has** a per-sample fit (`radius_uninduced.npz`, 4,155 cells) but was **excluded from
consensus seeding**, so its `isodepth_consensus` values are smoothed, not fit.

In [ ]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


## 1.1 What per-sample fits exist

In [ ]:
rows = []
for p in sorted(glob.glob(str(GP/"transfer_all"/"radius_*.npz"))) + sorted(glob.glob(str(GP/"isodepth_*.npz"))):
    z = np.load(p, allow_pickle=True)
    key = "radius" if "radius" in z.files else ("isodepth" if "isodepth" in z.files else z.files[0])
    rows.append(dict(file=os.path.basename(p), n_cells=len(z[key]), keys=",".join(z.files)))
F = pd.DataFrame(rows)
print(F.to_string(index=False))
verify("uninduced per-sample fit exists", bool((F.file=="radius_uninduced.npz").any()), True)
verify("uninduced fit n_cells", int(F.loc[F.file=="radius_uninduced.npz","n_cells"].iloc[0]), 4155, tol=0)

## 1.2 Which samples were seeded into the mouse consensus

Recorded: 5 samples seeded (uninduced excluded), n_seeded=20,342, n_unseeded=22,855.

In [ ]:
import scanpy as sc
a = sc.read_h5ad(ISO/"mouse_fibdec_isodepth.h5ad", backed="r")
tp = a.obs["timepoint"].astype(str)
print("isodepth-ish columns:", [c for c in a.obs.columns if "isodepth" in c])
# isodepth_quantile_raw = the SEEDED (fit) values only; isodepth_consensus = seeded + smoothed
raw = a.obs["isodepth_quantile_raw"].astype(float) if "isodepth_quantile_raw" in a.obs else None
con = a.obs["isodepth_consensus"].astype(float)
print("\nseeded (isodepth_quantile_raw) by timepoint:")
print(tp[raw.notna()].value_counts().to_string() if raw is not None else "column absent")
verify("n_seeded", int(raw.notna().sum()) if raw is not None else None, 20342, tol=0)
verify("n_unseeded", int(a.n_obs - raw.notna().sum()) if raw is not None else None, 22855, tol=0)
verify("uninduced cells in the FIT layer", int((raw.notna() & tp.str.contains("uninduced")).sum()), 0, tol=0,
       note="0 => uninduced values in isodepth_consensus are smoothed, not fit")
verify("uninduced cells in consensus layer", int((con.notna() & tp.str.contains("uninduced")).sum()), 9334, tol=0)

## 1.3 Geometric validation — does the fitted coordinate match tissue geometry?

Independent check: distance from the **myometrial boundary inward**, computed per tissue piece
(DBSCAN blobs; two sections contain two uterine-horn cross-sections each, so per-*section* geometry
is wrong). Recorded: **8 induced blobs, ρ 0.447–0.947, median 0.893**; uninduced 0.596–0.691.

The 0.447 blob (`1_dpi_late_mck12b` blob 1, 867 cells) is a known unexplained outlier — do not
quote the range without it.

In [ ]:
from scipy.stats import spearmanr
B = pd.read_csv(DATA/"boundary_depth_per_blob.csv")
rows=[]
for (tp_, s, bl), g in B.groupby(["timepoint","sample","blob"]):
    ok = g.dropna(subset=["depth_pct","isodepth_stored"])
    if len(ok) < 50: continue
    rows.append(dict(timepoint=tp_, sample=s, blob=bl, n=len(ok),
                     rho=round(float(spearmanr(ok.depth_pct, ok.isodepth_stored).statistic),4)))
V = pd.DataFrame(rows).sort_values(["timepoint","sample","blob"])
print(V.to_string(index=False))
ind = V[~V.timepoint.str.contains("uninduced")]
uni = V[ V.timepoint.str.contains("uninduced")]
print()
verify("induced blobs, n", len(ind), 8, tol=0)
verify("induced median rho", round(float(ind.rho.median()),3), 0.893)
verify("induced MIN rho (the outlier)", round(float(ind.rho.min()),3), 0.447,
       note="1_dpi_late_mck12b blob1, 867 cells - unexplained")
verify("induced max rho", round(float(ind.rho.max()),3), 0.947)
verify("uninduced median rho", round(float(uni.rho.median()),3), 0.655,
       note="lowest of any group - consistent with never being seeded")
verify("all induced blobs positively oriented", bool((ind.rho>0).all()), True,
       note="decidual-high convention preserved; no silent sign flip")

## Summary

In [ ]:
check_summary()